# Module 1: Agent Loop + Tools (12 min)

Build a customer service agent with tools, run it, and inspect the agent loop in action. By the end of this module you'll have a working agent that can look up customers, check orders, and process refunds.

**Prerequisites:** Python 3.10+, AWS credentials configured (Strands uses Bedrock by default)

---

## Part 1: Define Your Tools

Tools are Python functions decorated with `@tool`. The LLM reads the docstring to decide when to call them.

In [1]:
from strands import Agent, tool

# --- Mock customer data ---

CUSTOMERS = {
    "C-1001": {
        "name": "Sarah Johnson",
        "email": "sarah.johnson@email.com",
        "phone": "555-0142",
        "account_status": "active",
    },
    "C-1002": {
        "name": "Mike Chen",
        "email": "mike.chen@email.com",
        "phone": "555-0198",
        "account_status": "active",
    },
}

ORDERS = {
    "C-1001": [
        {"order_id": "ORD-5521", "item": "Wireless Headphones", "amount": 79.99,
         "status": "Delivered", "order_date": "2025-04-20", "delivered_date": "2025-04-28",
         "tracking": "TRK-998877"},
        {"order_id": "ORD-5488", "item": "USB-C Hub", "amount": 45.00,
         "status": "Shipped", "order_date": "2025-05-01", "estimated_delivery": "2025-05-06",
         "tracking": "TRK-887766"},
    ],
    "C-1002": [
        {"order_id": "ORD-5390", "item": "Mechanical Keyboard", "amount": 149.99,
         "status": "Delayed", "order_date": "2025-04-15", "estimated_delivery": "2025-04-25",
         "tracking": "TRK-776655"},
    ],
}


@tool
def lookup_customer(customer_id: str) -> str:
    """Look up a customer by their ID.

    Args:
        customer_id: The customer ID (e.g. C-1001)
    """
    customer = CUSTOMERS.get(customer_id)
    if not customer:
        return f"No customer found with ID {customer_id}"
    return (
        f"Customer: {customer['name']}\n"
        f"Email: {customer['email']}\n"
        f"Phone: {customer['phone']}\n"
        f"Account Status: {customer['account_status']}"
    )


@tool
def get_order_history(customer_id: str) -> str:
    """Get order history for a customer.

    Args:
        customer_id: The customer ID (e.g. C-1001)
    """
    orders = ORDERS.get(customer_id)
    if not orders:
        return f"No orders found for customer {customer_id}"
    lines = []
    for order in orders:
        line = (
            f"Order {order['order_id']}: {order['item']} — ${order['amount']:.2f} "
            f"[{order['status']}] Ordered: {order['order_date']} "
        )
        if order.get("delivered_date"):
            line += f"Delivered: {order['delivered_date']} "
        if order.get("estimated_delivery"):
            line += f"Est. Delivery: {order['estimated_delivery']} "
        line += f"Tracking: {order['tracking']}"
        lines.append(line)
    return "\n".join(lines)


@tool
def process_refund(order_id: str, amount: float) -> str:
    """Process a refund for an order.

    Args:
        order_id: The order ID to refund
        amount: The refund amount in dollars
    """
    return f"Refund of ${amount:.2f} processed for order {order_id}. Expect 3-5 business days."


print("✅ Tools defined: lookup_customer, get_order_history, process_refund")

✅ Tools defined: lookup_customer, get_order_history, process_refund


---

## Part 2: Create and Run the Agent

Wire the tools into an `Agent` with a system prompt. Then ask it to help a customer.

In [2]:
SYSTEM_PROMPT = """You are a customer service agent for an online electronics store.
Be helpful, professional, and concise. Use the available tools to look up customer
information and process requests.

Important guidelines:
- Always verify the customer using lookup_customer before taking action.
- Use tool data to answer questions — don't ask the customer for info you already have.
- Be warm but efficient."""

agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    system_prompt=SYSTEM_PROMPT,
)

# Run the agent
result = agent("Hi, I'm customer C-1001. Can you check on my recent orders?")

Sure! Let me pull up your account and order history right away.
Tool #1: lookup_customer

Tool #2: get_order_history
Hi Sarah! Here's a summary of your recent orders:

1. **ORD-5521 – Wireless Headphones** ($79.99)
   - ✅ Delivered on April 28, 2025
   - Tracking: TRK-998877

2. **ORD-5488 – USB-C Hub** ($45.00)
   - 🚚 Shipped – Estimated delivery: May 6, 2025
   - Tracking: TRK-887766

Is there anything else I can help you with regarding these orders?

---

## Part 3: Inspect the Agent Loop

The agent loop is: **User → LLM → Tool Call → Tool Result → LLM → Response**. Let's see it in the message history.

In [ ]:
import json

print(f"Total messages in conversation: {len(agent.messages)}")
print("=" * 60)

for i, msg in enumerate(agent.messages):
    role = msg["role"]
    if role == "user":
        text = msg["content"][0].get("text", "") if msg["content"] else ""
        print(f"\n[{i}] 👤 USER: {text[:80]}")
    elif role == "assistant":
        for block in msg.get("content", []):
            if "text" in block:
                print(f"\n[{i}] 🤖 ASSISTANT: {block['text'][:100]}...")
            elif "toolUse" in block:
                tu = block["toolUse"]
                print(f"\n[{i}] 🔧 TOOL CALL: {tu['name']}({json.dumps(tu.get('input', {}))})")

print("\n" + "=" * 60)
print(f"\n📊 Metrics:")
print(f"   Cycles: {result.metrics.get_summary()['total_cycles']}")
print(f"   Tokens: {result.metrics.accumulated_usage['totalTokens']}")

---

## 🎯 Try It Yourself

Try different customer scenarios:

In [ ]:
# Try these prompts:
# agent("I'm C-1002. My keyboard order is delayed — what's going on?")
# agent("I'd like a refund for order ORD-5521 please.")
# agent("Look up customer C-9999 for me.")  # Non-existent customer

agent("I'm C-1002. My keyboard order is delayed — what's going on?")

---

## 💬 Want a real multi-turn conversation?

In a notebook, each cell is a **single turn** — you call `agent("...")` once and read the reply. To chat back and forth (the agent remembering earlier turns), run the companion script in a **terminal**, which wraps this same agent in a loop. From the cloned repo:

```bash
cd samples/01-agent-loop-tools
pip install -r requirements.txt
python chat.py
```

Type your messages, and `quit` (or Ctrl+C) to exit. The agent keeps its context across turns because it reuses one `Agent` instance — its history lives in `agent.messages`.

---

## What's Next

The agent works, but it has no guardrails. In **Module 2: Hooks**, you'll add a rate limiter that caps tool calls — deterministic code intercepting the agent loop.